In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Configuration

**`METRIC`** — the error metric used for:
- best-seed selection per product/model (lower is better)
- ranking hybrids against the baseline
- computing % improvement (positive = hybrid improves over baseline)

Set `METRIC = 'mae'` (default) or swap to `'rmse'` to switch the entire analysis.

In [2]:
METRIC = 'mae'   # swap to 'rmse' to use RMSE throughout

In [3]:
import pandas as pd


# 1. Load your dataset
df = pd.read_csv('spearman_inference_fix.csv')

print(df['product_id'].unique())
print(len(df['product_id'].unique()))

# 1b. Drop the graph-free ablation rows (ablate_z == True).
#     For gcn_lstm / gcn_mlp the grid also runs an ablation where the graph
#     embedding z is zeroed; that model is a plain LSTM/MLP (AblationLSTMForecaster
#     / AblationMLPForecaster) and reproduces the baseline. Keeping it lets the
#     best-seed selection pick a graph-free run as the "best gcn_lstm/gcn_mlp",
#     creating artificial ties with the baseline.
#     NOTE: use `!= True`, NOT `== False`. Baseline rows carry a blank ablate_z
#     (parsed as NaN); `== False` would also drop every baseline.
df = df[df['ablate_z'] != True]
df = df.drop(columns=['ablate_z'])

[   278  18568  26372 110187 213626 213633 545597 606230 908786 919137
    457  25032  26768 151815 213627 213636 545598 680152 911753 921558
   1325  26002  26924 151821 213628 213642 545602 904144 912509 932812
   6032  26008  30534 151828 213629 500056 582893 907967 915640 933442
  13544  26076  48369 213624 213630 500057 587317 907969 916110 933457
  16920  26220  48375 213625 213631 538923 606214 907970 918537 936693]
60


In [4]:
import pandas as pd

# NOTE: ablation rows (ablate_z == True) were already dropped upstream, so every
# model_variant below is the genuine model (the gcn_* rows are the real graph
# models, never the graph-free ablation that mimics the baseline).

# 2. For each product/model, pick the BEST SEED — the run with the lowest METRIC value.
best_seed_idx = df.groupby(['product_id', 'model_variant'])[METRIC].idxmin()
best_seed = df.loc[best_seed_idx]

# Values carried over from that best seed of each product/model
metrics = ['rmse', 'mae', 'bias', 'pocid', 'seed']

# 3. Pivot so each row represents exactly one product
pivot_df = best_seed.pivot(index='product_id', columns='model_variant', values=metrics)

# Flatten the MultiIndex columns: 'mae_lstm_baseline', 'seed_gcn_lstm', etc.
pivot_df.columns = [f"{metric}_{model}" for metric, model in pivot_df.columns]
pivot_df = pivot_df.reset_index()

# Define all available models to search through
all_models = [
    'lstm_baseline', 'graph2vec_lstm', 'gcn_lstm', 'gat_lstm',
    'mlp_baseline', 'graph2vec_mlp', 'gcn_mlp', 'gat_mlp'
]

# 4. Extract the absolute best model per product (using METRIC)
best_models_per_product = []

for _, row in pivot_df.iterrows():
    prod_id = int(row['product_id'])

    best_model_name = min(all_models, key=lambda m: row[f'{METRIC}_{m}'])

    record = {
        'product_id': prod_id,
        'best_overall_model': best_model_name,
        'is_hybrid': best_model_name not in ['lstm_baseline', 'mlp_baseline'],
        'best_model_seed': int(row[f'seed_{best_model_name}']),
        'best_model_rmse': row[f'rmse_{best_model_name}'],
        'best_model_mae': row[f'mae_{best_model_name}'],
        'best_model_bias': row[f'bias_{best_model_name}'],
        'best_model_pocid': row[f'pocid_{best_model_name}'],
    }

    for m in all_models:
        record[f'{m}_{METRIC}'] = row[f'{METRIC}_{m}']
        record[f'{m}_seed']     = int(row[f'seed_{m}'])

    best_models_per_product.append(record)

results_df = pd.DataFrame(best_models_per_product)
results_df.to_csv(f'absolute_best_models_with_all_{METRIC}s.csv', index=False)

print(f"Metric used for seed selection: {METRIC}")
print(f"Processed {len(results_df)} products.")
print("\nDistribution of winning models across the dataset:")
print(results_df['best_overall_model'].value_counts())
print(f"\nResults saved to 'absolute_best_models_with_all_{METRIC}s.csv'")

Metric used for seed selection: mae
Processed 60 products.

Distribution of winning models across the dataset:
best_overall_model
lstm_baseline     13
mlp_baseline      12
gcn_lstm          10
graph2vec_lstm     9
graph2vec_mlp      7
gat_lstm           6
gcn_mlp            2
gat_mlp            1
Name: count, dtype: int64

Results saved to 'absolute_best_models_with_all_maes.csv'


In [5]:
import pandas as pd

lstm_hybrids = ['graph2vec_lstm', 'gcn_lstm', 'gat_lstm']
mlp_hybrids  = ['graph2vec_mlp',  'gcn_mlp',  'gat_mlp']


def compare_hybrids_to_baseline(results_df, hybrids, baseline):
    """
    For every product, compute per-hybrid pct improvement over the baseline.
    pct_improvement_{h} = (baseline - h) / baseline * 100
      positive → hybrid is better (lower error)
      negative → hybrid is worse
    """
    out = []
    for _, row in results_df.iterrows():
        base_val = row[f'{baseline}_{METRIC}']

        record = {
            'product_id':         int(row['product_id']),
            'baseline':           baseline,
            f'baseline_{METRIC}': base_val,
            'baseline_seed':      int(row[f'{baseline}_seed']),
        }

        for h in hybrids:
            h_val = row[f'{h}_{METRIC}']
            record[f'{h}_{METRIC}']        = h_val
            record[f'{h}_seed']            = int(row[f'{h}_seed'])
            record[f'{h}_pct_improvement'] = (base_val - h_val) / base_val * 100

        best_h = min(hybrids, key=lambda h: row[f'{h}_{METRIC}'])
        record['best_hybrid']                  = best_h
        record[f'best_hybrid_{METRIC}']        = row[f'{best_h}_{METRIC}']
        record['best_hybrid_beats_baseline']   = bool(row[f'{best_h}_{METRIC}'] < base_val)
        record['best_hybrid_pct_improvement']  = record[f'{best_h}_pct_improvement']

        out.append(record)

    return pd.DataFrame(out)


lstm_vs_baseline = compare_hybrids_to_baseline(results_df, lstm_hybrids, 'lstm_baseline')
mlp_vs_baseline  = compare_hybrids_to_baseline(results_df, mlp_hybrids,  'mlp_baseline')

for name, table in [(f'lstm_hybrid_vs_baseline_{METRIC}', lstm_vs_baseline),
                    (f'mlp_hybrid_vs_baseline_{METRIC}',  mlp_vs_baseline)]:
    table.to_csv(f'{name}.csv',   index=False)
    table.to_excel(f'{name}.xlsx', index=False)

print(f"Metric used: {METRIC}\n")
for label, table, hybrids in [
    ('LSTM', lstm_vs_baseline, lstm_hybrids),
    ('MLP',  mlp_vs_baseline,  mlp_hybrids),
]:
    print(f"=== {label} hybrids vs baseline ({len(table)} products) ===")
    for h in hybrids:
        beats = (table[f'{h}_pct_improvement'] > 0).sum()
        avg   = table[f'{h}_pct_improvement'].mean()
        print(f"  {h:20s}: beats baseline in {beats:2d}/60 products | avg pct improvement = {avg:+.2f}%")
    print()

Metric used: mae

=== LSTM hybrids vs baseline (60 products) ===
  graph2vec_lstm      : beats baseline in 29/60 products | avg pct improvement = -1.97%
  gcn_lstm            : beats baseline in 22/60 products | avg pct improvement = -2.88%
  gat_lstm            : beats baseline in 23/60 products | avg pct improvement = -4.99%

=== MLP hybrids vs baseline (60 products) ===
  graph2vec_mlp       : beats baseline in 23/60 products | avg pct improvement = -1.83%
  gcn_mlp             : beats baseline in 19/60 products | avg pct improvement = -3.76%
  gat_mlp             : beats baseline in 20/60 products | avg pct improvement = -3.32%



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

lstm_hybrids = ['graph2vec_lstm', 'gcn_lstm', 'gat_lstm']
mlp_hybrids  = ['graph2vec_mlp',  'gcn_mlp',  'gat_mlp']
product_ids  = pivot_df['product_id'].astype(int).values

CHUNK_SIZE = 30  # products per figure

def pct_improvement(pivot_df, hybrids, baseline, metric):
    base_col = pivot_df[f'{metric}_{baseline}'].values
    data = {}
    for h in hybrids:
        h_col = pivot_df[f'{metric}_{h}'].values
        data[h] = (base_col - h_col) / base_col * 100
    return pd.DataFrame(data, index=product_ids)

def draw_heatmap_chunk(matrix_chunk, title, note, fname, abs_max):
    matrix_chunk.index.name = 'Product ID'
    n_rows = len(matrix_chunk)

    fig, ax = plt.subplots(figsize=(7, max(6, n_rows * 0.38)))
    sns.heatmap(
        matrix_chunk,
        ax=ax,
        cmap='RdYlGn',
        center=0,
        vmin=-abs_max,
        vmax=abs_max,
        annot=True,
        fmt='.1f',
        linewidths=0.3,
        linecolor='lightgrey',
        cbar_kws={'label': f'% improvement over baseline\n({note})', 'shrink': 0.6},
        annot_kws={'size': 7},
    )
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Model', fontsize=9)
    ax.set_ylabel('Product ID', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', labelsize=7)
    plt.tight_layout()
    for ext in ('png', 'pdf'):
        plt.savefig(fname.replace('.png', f'.{ext}'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {fname.replace('.png', '.png/.pdf')}")

def draw_split_heatmaps(matrix, base_title, note, base_fname):
    # Use a single shared abs_max across all chunks for consistent colour scale
    abs_max = max(abs(matrix.values.min()), abs(matrix.values.max()))
    chunks = [matrix.iloc[i:i+CHUNK_SIZE] for i in range(0, len(matrix), CHUNK_SIZE)]
    for idx, chunk in enumerate(chunks, start=1):
        part_label = f"Part {idx}/{len(chunks)}"
        draw_heatmap_chunk(
            chunk,
            title=f"{base_title}\n({part_label})",
            note=note,
            fname=base_fname.replace('.png', f'_part{idx}.png'),
            abs_max=abs_max,
        )

# ── LSTM MAE ──────────────────────────────────────────────────────────────────
draw_split_heatmaps(
    pct_improvement(pivot_df, lstm_hybrids, 'lstm_baseline', 'mae'),
    'LSTM — MAE % Relative Improvement over Baseline\n(positive = lower error)',
    'positive = lower MAE than lstm_baseline',
    'heatmap_lstm_mae.png',
)

# ── LSTM POCID ────────────────────────────────────────────────────────────────
draw_split_heatmaps(
    pct_improvement(pivot_df, lstm_hybrids, 'lstm_baseline', 'pocid'),
    'LSTM — POCID % Relative Improvement over Baseline\n(positive = higher directional accuracy)',
    'positive = higher POCID than lstm_baseline',
    'heatmap_lstm_pocid.png',
)

# ── MLP MAE ───────────────────────────────────────────────────────────────────
draw_split_heatmaps(
    pct_improvement(pivot_df, mlp_hybrids, 'mlp_baseline', 'mae'),
    'MLP — MAE % Relative Improvement over Baseline\n(positive = lower error)',
    'positive = lower MAE than mlp_baseline',
    'heatmap_mlp_mae.png',
)

# ── MLP POCID ─────────────────────────────────────────────────────────────────
draw_split_heatmaps(
    pct_improvement(pivot_df, mlp_hybrids, 'mlp_baseline', 'pocid'),
    'MLP — POCID % Relative Improvement over Baseline\n(positive = higher directional accuracy)',
    'positive = higher POCID than mlp_baseline',
    'heatmap_mlp_pocid.png',
)
